# ServiceIQ — RAG Assistant for Field Technicians (OpenAI)

One Gradio app, four tabs, four GenAI frameworks — all on the same OpenAI models:

| Tab | Framework | What it solves | Retrieval? |
|---|---|---|---|
| 1 | **LangChain** | Troubleshooting assistant (LLM only, LCEL chain) | ❌ none |
| 2 | **LlamaIndex** | Manual Q&A from a PDF index (top-2 sources shown) | ✅ indexing + query |
| 3 | **Semantic Kernel** | Maintenance planner plugin (semantic + native functions) | ❌ truncated manual text |
| 4 | **Haystack** | Grounded RAG pipeline (explicit stages, wired with `connect()`) | ✅ full pipeline, refuses out-of-scope |

**Two phases** (the core RAG idea):
- **Indexing phase** (run once per manual): extract → chunk → embed → store
- **Query phase** (run per question): embed query → retrieve → augment prompt → generate

**Grounding rule** (Tabs 2 and 4): answer *only* from the retrieved context; if the context
does not contain the answer, say **"I don't know."** This is what makes the assistant safe
for proprietary equipment documentation — a bare LLM (Tab 1) cannot make that promise.

**Config:** chat model is `gpt-5.4-mini`, kept in one `OPENAI_MODEL` variable; embeddings are
`text-embedding-3-small`; the API key is entered with `getpass` and never written in code.

**Every tab** has a live execution panel (pipeline steps light up as they run) and an execution log.

---
## Task 0 — Setup and shared helpers

In [ ]:
%pip install -q langchain-openai llama-index-llms-openai llama-index-embeddings-openai semantic-kernel haystack-ai gradio pypdf fpdf2

In [ ]:
# --- Configuration: one place to change the model -------------------------
import os
import getpass

OPENAI_MODEL = "gpt-5.4-mini"                # chat model (single switch)
OPENAI_EMBED_MODEL = "text-embedding-3-small"  # embedding model

# The key is read interactively and stored only in the environment.
# It is never hard-coded, never printed, and never saved in this notebook.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OPENAI_API_KEY: ")

print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("Chat model:    ", OPENAI_MODEL)
print("Embed model:   ", OPENAI_EMBED_MODEL)

In [ ]:
# --- Shared helper 1: PDF text extraction (pypdf) --------------------------
from pypdf import PdfReader

def extract_pdf_text(pdf_path: str) -> str:
    """Extract all text from a PDF, keeping page markers so answers stay traceable."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append(f"[Page {i}]\n{text.strip()}")
    return "\n\n".join(pages)

In [ ]:
# --- Shared helpers 2 + 3: execution panel and log -------------------------
import html as _html
from datetime import datetime

def render_pipeline_html(steps, active_index=-1, done_upto=0):
    """Render the pipeline steps as chips: done (green), active (pulsing), pending (grey)."""
    chips = []
    for i, step in enumerate(steps):
        if i < done_upto:
            cls, icon = "sp-done", "&#10003;"
        elif i == active_index:
            cls, icon = "sp-active", "&#9654;"
        else:
            cls, icon = "sp-pending", "&#9675;"
        chips.append(
            f'<div class="{cls}"><span class="sp-icon">{icon}</span>'
            f'<span>{_html.escape(str(step))}</span></div>'
        )
    return (
        '<style>'
        '.pipeline{display:flex;flex-wrap:wrap;gap:6px;margin:4px 0;}'
        '.pipeline>div{display:flex;align-items:center;gap:6px;padding:6px 10px;'
        'border-radius:14px;font:13px/1.3 sans-serif;border:1px solid #d0d4dc;background:#f6f7f9;color:#5a6072;}'
        '.sp-done{background:#e5f6ec;border-color:#9fd8b4;color:#116632;}'
        '.sp-active{background:#fff4d6;border-color:#e6b84c;color:#7a5a00;font-weight:600;}'
        '.sp-icon{font-size:11px;}'
        '</style>'
        f'<div class="pipeline">{"".join(chips)}</div>'
    )

def append_log(log: str, message: str) -> str:
    """Append a timestamped line to the execution log (returns the new full log)."""
    stamp = datetime.now().strftime("%H:%M:%S")
    return (log + f"\n[{stamp}] {message}").strip()

In [ ]:
# --- Sample equipment manual (fictional, generated locally as a PDF) -------
# A combined service volume covering three product lines. It deliberately does
# NOT contain any warranty terms and no "moon rover", so the out-of-scope query
# ("What is the warranty period for the moon rover?") must be refused.
from fpdf import FPDF

MANUAL_PDF = "serviceiq_sample_manual.pdf"

MANUAL_SECTIONS = [
("ServiceIQ Field Equipment Manual - Combined Service Volume (Fictional Sample)",
 "This volume covers the DX-500 fuel dispenser, the CR-220 refrigerated display case "
 "and the P-100 diaphragm pump. All procedures must be performed by certified field "
 "technicians. Reference this manual before any intervention and record every service "
 "event in the work-order system within 24 hours."),

("Section 1 - Fuel Dispenser DX-500",
 "The DX-500 is a submersible-turbine fuel dispenser for retail forecourts. It is "
 "controlled by a pump controller board and a flow meter head that reports pulses over "
 "a shielded cable to junction J4. The dispenser runs on 220 to 240 volts AC single "
 "phase and must always be earthed. Before opening the cabinet, switch off the main "
 "breaker and confirm the display is dark."),

("1.1 Error codes",
 "E-101 No communication with pump controller. Check the CAN bus termination resistor "
 "and reseat the controller ribbon cable. "
 "E-104 Flow meter calibration fault detected after a power interruption. This is the "
 "most common post-power-cycle fault. Proceed in this order. First, verify the main "
 "power supply voltage is within 220 to 240 volts AC and that the earth connection is "
 "tight. Second, inspect the flow meter pulse cable connector at junction J4 for "
 "corrosion, bent pins or a loose latch; reseat the connector firmly. Third, perform a "
 "controlled power cycle: switch off the main breaker, wait a full 60 seconds, restore "
 "power, and allow the unit 90 seconds to reboot before judging the result. Fourth, if "
 "E-104 returns within 10 minutes, run the flow meter calibration routine from the "
 "technician menu: Menu, Service, Calibration, Flow Meter. The routine requires a "
 "20-litre certified verification canister and takes about five minutes. Fifth, if the "
 "fault still persists after recalibration, replace the pulse sensor board, part number "
 "DX-PM-114, and repeat the calibration. "
 "E-120 Vapor recovery fan overcurrent. Clean the fan grille and check the fan fuse."),

("1.2 Controlled power-cycle procedure",
 "A controlled power cycle protects the flow meter head. Switch off the main breaker. "
 "Wait 60 seconds so the controller capacitors fully discharge. Restore power. Wait 90 "
 "seconds for the boot sequence. Do not interrupt power during boot; interrupting the "
 "boot is itself a known trigger of error E-104."),

("1.3 Preventive maintenance - DX-500",
 "Monthly: inspect nozzles, breakaway couplings and hoses for wear. Quarterly: replace "
 "the inlet filter element and drain the sump water separator. Annually: verify flow "
 "meter calibration against the certified canister and torque-check the cabinet "
 "fasteners."),

("Section 2 - Refrigerated Display Case CR-220",
 "The CR-220 is a multi-deck refrigerated display case for retail environments. It uses "
 "an electric defrost heater controlled by the case controller and holds perishable "
 "goods at 2 to 4 degrees Celsius."),

("2.1 Defrost settings",
 "The CR-220 runs automatic electric defrost cycles. The recommended defrost interval "
 "is every 6 hours, with a defrost duration of 20 minutes per cycle. In stores with "
 "high humidity or heavy door traffic, shorten the interval to every 4 hours to prevent "
 "coil icing. A full manual defrost is recommended once per month. To change the "
 "interval, open the controller menu, select Settings, Defrost, Interval, and confirm "
 "with the lock key. Never set the interval below 4 hours; heater life is reduced."),

("2.2 Routine care",
 "Clean the condenser coil every 30 days with a soft brush and vacuum. Inspect door "
 "gaskets monthly and replace any gasket that no longer seals. Check the drain line and "
 "heated drain trap quarterly to prevent water pooling under the case."),

("2.3 Error codes",
 "F-07 High superheat alarm. Check the TXV bulb strap and refrigerant charge. "
 "F-12 Evaporator icing suspected. Run a manual defrost and re-check within one hour."),

("Section 3 - Industrial Pump P-100 (Air-Operated Double-Diaphragm)",
 "The P-100 moves chemical fluids on plant sites. It has two PTFE diaphragms driven by "
 "compressed air. Diaphragm replacement is a routine workshop task but the pump must "
 "always be brought to zero energy state first."),

("3.1 Lockout/tagout before replacing the pump diaphragm",
 "Perform these steps in order and do not skip any of them. Step 1: notify operations "
 "that the pump is going out of service. Step 2: close and lock the air supply valve in "
 "the closed position and apply your personal lock. Step 3: bleed the air pressure by "
 "opening the air bleed valve until the supply gauge reads 0 bar. Step 4: close the "
 "process inlet and outlet valves, disconnect the process lines and cap them to prevent "
 "residual backflow. Step 5: attach your personal tag reading DO NOT OPERATE to the "
 "locked air valve. Step 6: verify zero energy by pressing the local start button; the "
 "pump must not actuate. Step 7: put on chemical-resistant gloves and a face shield "
 "before opening any wetted part, because residual fluid may be hazardous."),

("3.2 Diaphragm replacement",
 "After lockout/tagout is verified: remove the outer fluid chamber bolts, withdraw the "
 "outer piston plate, remove the old diaphragm and inspect the seat for scoring. "
 "Install the new PTFE diaphragm, part number P-100-DM-08, with the moulded centre disc "
 "facing the air side. Torque the centre bolt to 24 newton-metres and the chamber bolts "
 "to 18 newton-metres in a cross pattern. Reconnect the process lines, remove locks and "
 "tags, restore air slowly and leak-check at 50 percent duty before returning to "
 "service."),

("3.3 Preventive maintenance - P-100",
 "Every 90 days inspect both diaphragms for swelling, crazing or puncture. Replace both "
 "diaphragms every 12 months regardless of appearance. Check air valve lubrication "
 "monthly and keep the muffler clean."),

("Section 4 - General safety",
 "Only certified technicians may perform the procedures in this manual. Never bypass a "
 "safety interlock or a pressure relief device. Always confirm zero energy state before "
 "working on any energised or pressurised equipment. Record every service event, "
 "including error codes observed and parts replaced, in the work-order system within 24 "
 "hours of completion."),
]

def build_sample_manual(path: str = MANUAL_PDF) -> str:
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=18)
    pdf.set_margins(20, 18, 20)
    for title, body in MANUAL_SECTIONS:
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 14)
        pdf.multi_cell(0, 8, title)
        pdf.ln(2)
        pdf.set_font("Helvetica", "", 11)
        pdf.multi_cell(0, 6, body)
    pdf.output(path)
    return path

build_sample_manual()
print("Sample manual written to:", MANUAL_PDF)

In [ ]:
# --- Sanity check: extract the text back -----------------------------------
manual_text = extract_pdf_text(MANUAL_PDF)
print("Pages:", manual_text.count("[Page"))
print("Words:", len(manual_text.split()))
print("Characters:", len(manual_text))
print("\n----- first 600 characters -----")
print(manual_text[:600])

---
## Task 1 — LangChain: troubleshooting assistant (LLM only)

A minimal LCEL chain: `prompt | ChatOpenAI | StrOutputParser`. **No retrieval** — the
model only gets its field-service role and the question. Each component is invoked
separately inside a generator callback so the panel shows: User Input → PromptTemplate →
LLM → Output Parser → Response.

> ⚠️ **Why this tab can still hallucinate** (report note): the model never sees the
> manual. It answers from generic pretraining knowledge, so it can invent plausible but
> wrong steps, cite error codes that do not exist in this product line, or mix
> procedures across models and firmware versions. For safety-critical repair steps this
> is exactly the risk RAG removes: with retrieval, every claim can be traced to a
> retrieved passage, and a grounding prompt can force an "I don't know" when the context
> does not support an answer.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

lc_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2)

lc_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        "You are a senior field-service engineer for Dover operating-company equipment "
        "(fuel dispensers, commercial refrigeration, industrial pumps).\n"
        "Answer the technician's question with clear, numbered troubleshooting steps.\n"
        "Be practical and safety-first: mention lockout/tagout or power isolation when "
        "relevant.\n"
        "If you are not certain about model-specific details, say so explicitly.\n\n"
        "Technician question: {question}"
    ),
)

lc_parser = StrOutputParser()

lc_chain = lc_prompt | lc_llm | lc_parser
print("LCEL chain:", lc_chain)

In [ ]:
LC_STEPS = ["User Input", "PromptTemplate", "LLM", "Output Parser", "Response"]

def run_langchain_tab(question, panel_html, log):
    """Generator: yields (panel_html, log, answer) so Gradio streams each step."""
    if not question or not question.strip():
        yield panel_html, append_log(log, "No question entered."), ""
        return

    log = append_log(log, f"Tab 1 (LangChain) received: {question!r}")
    yield render_pipeline_html(LC_STEPS, active_index=0, done_upto=0), log, ""

    # Step 2 — PromptTemplate
    prompt_value = lc_prompt.invoke({"question": question.strip()})
    log = append_log(log, f"PromptTemplate rendered ({len(prompt_value.text)} chars)")
    yield render_pipeline_html(LC_STEPS, active_index=1, done_upto=1), log, ""

    # Step 3 — LLM
    log = append_log(log, f"Calling ChatOpenAI ({OPENAI_MODEL}) ...")
    yield render_pipeline_html(LC_STEPS, active_index=2, done_upto=2), log, ""
    llm_result = lc_llm.invoke(prompt_value.to_messages())

    # Step 4 — Output Parser
    log = append_log(log, f"LLM returned {len(llm_result.content)} chars")
    yield render_pipeline_html(LC_STEPS, active_index=3, done_upto=3), log, ""
    answer = lc_parser.invoke(llm_result)

    # Step 5 — Response (also verify the full LCEL chain gives the same result)
    chain_answer = lc_chain.invoke({"question": question.strip()})
    log = append_log(log, "Step-by-step invoke == full LCEL chain invoke: "
                          f"{chain_answer.strip() == answer.strip()}")
    log = append_log(log, "Done. NOTE: no retrieval was used — answer may hallucinate.")
    yield render_pipeline_html(LC_STEPS, active_index=-1, done_upto=len(LC_STEPS)), log, answer

print("Tab 1 handler ready.")

---
## Task 2 — LlamaIndex: manual Q&A with sources

`Settings.llm` and `Settings.embed_model` are set once; the PDF text becomes a
`Document`, and a `VectorStoreIndex` is built **once** (Build Index button). Each
question then goes through the query engine with `similarity_top_k=2`, and the
`source_nodes` are displayed next to the answer — so every claim is traceable to a
passage.

Two buttons keep the two RAG phases explicit: **Build Index** (indexing phase) and
**Ask** (query phase).

In [ ]:
from llama_index.core import Settings, Document, VectorStoreIndex
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model=OPENAI_MODEL, temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model=OPENAI_EMBED_MODEL)

li_index = None
li_query_engine = None

LI_INDEX_STEPS = ["PDF Text", "Document", "Embed & Store (VectorStoreIndex)"]
LI_QUERY_STEPS = ["Question", "Embed Query", "Retrieve top-2", "Generate Answer", "Show Sources"]

def build_li_index(pdf_path=None):
    """Indexing phase — run once per manual."""
    global li_index, li_query_engine
    path = pdf_path or MANUAL_PDF
    text = extract_pdf_text(path)
    doc = Document(text=text, metadata={"source": path, "product": "ServiceIQ sample manual"})
    li_index = VectorStoreIndex.from_documents([doc])
    li_query_engine = li_index.as_query_engine(similarity_top_k=2)
    return (f"Index built from {path}: {len(text.split())} words, "
            f"{len(text) // 4000 + 1} implicit chunks. similarity_top_k=2.")

def ask_li(question, log):
    """Query phase — embed query, retrieve top-2, generate, show sources."""
    global li_query_engine
    if li_query_engine is None:
        yield render_pipeline_html(LI_QUERY_STEPS), append_log(log, "Blocked: build the index first."), ""
        return
    if not question or not question.strip():
        yield render_pipeline_html(LI_QUERY_STEPS), append_log(log, "No question entered."), ""
        return

    q = question.strip()
    log = append_log(log, f"Tab 2 (LlamaIndex) received: {q!r}")
    yield render_pipeline_html(LI_QUERY_STEPS, active_index=0, done_upto=0), log, ""

    log = append_log(log, "Embedding query and retrieving top-2 chunks ...")
    yield render_pipeline_html(LI_QUERY_STEPS, active_index=2, done_upto=2), log, ""

    response = li_query_engine.query(q)

    sources = []
    for i, node in enumerate(response.source_nodes, start=1):
        score = f"{node.score:.3f}" if node.score is not None else "n/a"
        sources.append(f"**Source {i}** (score {score}):\n\n{node.text}")
    sources_text = "\n\n---\n\n".join(sources)

    log = append_log(log, f"Retrieved {len(response.source_nodes)} source nodes")
    yield render_pipeline_html(LI_QUERY_STEPS, active_index=4, done_upto=len(LI_QUERY_STEPS)), log, \
          f"### Answer\n\n{response}\n\n### Retrieved sources\n\n{sources_text}"

print("Tab 2 handlers ready. Run build_li_index() or use the Build Index button.")

---
## Task 3 — Semantic Kernel: maintenance planner plugin

One plugin, two kinds of function:

- **`generate_maintenance_schedule(material_text, num_days)`** — a *semantic* function:
  a chat prompt (registered with `kernel.add_function`, `template_format="semantic-kernel"`)
  whose `<message role="...">` blocks are rendered into a **`ChatHistory`** and sent to
  `OpenAIChatCompletion`. It asks for a day-wise preventive-maintenance plan.
- **`mock_work_order_sync(plan_text)`** — a *native* function (`@kernel_function`): pure
  Python that turns every **"Day X"** line of the plan into a mock work order. No real
  system is touched.

The manual text is truncated to 3,000 characters (per the reference table) so the prompt
stays small. A slider sets the maintenance horizon (1–7 days).

In [ ]:
import re
import asyncio
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function, KernelArguments
from semantic_kernel.contents import ChatHistory

sk_kernel = Kernel()
sk_kernel.add_service(
    OpenAIChatCompletion(service_id="default", ai_model_id=OPENAI_MODEL)
)

MAINTENANCE_PROMPT = """
<message role="system">
You are ServiceIQ's preventive-maintenance planner for Dover operating-company equipment.
Base the plan ONLY on the manual excerpt provided. If the excerpt does not describe a
task clearly, keep the plan generic and safe. Start every plan line with "Day N:".
</message>
<message role="user">
Manual excerpt (may be truncated):
{{$material_text}}

Create a day-wise preventive-maintenance plan for the next {{$num_days}} day(s).
One section per day, formatted exactly as:
Day 1: <tasks for that day>
Day 2: <tasks for that day>
Keep each day to 2-3 short tasks.
</message>
"""

# Semantic function, registered under the same plugin name.
sk_schedule_fn = sk_kernel.add_function(
    plugin_name="MaintenancePlanner",
    function_name="generate_maintenance_schedule",
    prompt=MAINTENANCE_PROMPT,
    template_format="semantic-kernel",
    description="Generates a day-wise preventive-maintenance plan from the manual text.",
)

class MaintenancePlannerPlugin:
    @kernel_function(
        name="mock_work_order_sync",
        description="Turns each 'Day X' line of a maintenance plan into a mock work order.",
    )
    def mock_work_order_sync(self, plan_text: str) -> str:
        orders = []
        for line in plan_text.splitlines():
            m = re.match(r"\s*(?:\*\*)?\s*Day\s*(\d+)\s*[:\-]\s*(.+)", line.strip())
            if m:
                day, task = m.group(1), m.group(2).strip().strip("*")
                orders.append(
                    f"WO-{len(orders) + 1:04d} | Day {day} | status=OPEN "
                    f"| system=MOCK (no real CMMS) | task={task}"
                )
        if not orders:
            return "No 'Day X' lines found — no work orders created."
        return "\n".join(orders)

sk_kernel.add_plugin(MaintenancePlannerPlugin(), plugin_name="MaintenancePlanner")
print("Registered functions in plugin 'MaintenancePlanner':")
print(" -", sk_schedule_fn.fully_qualified_name, "(semantic)")
print(" - MaintenancePlanner-mock_work_order_sync (native)")

In [ ]:
SK_STEPS = ["Manual Text (truncated)", "Kernel + ChatHistory",
            "Semantic: generate_maintenance_schedule", "Native: mock_work_order_sync", "Work Orders"]

async def run_sk_tab(num_days, panel_html, log):
    """Async generator: Gradio streams SK's async pipeline step by step."""
    if not (1 <= int(num_days) <= 7):
        yield panel_html, append_log(log, "Pick a horizon between 1 and 7 days."), ""
        return

    material_text = extract_pdf_text(MANUAL_PDF)[:3000]   # truncated per reference table
    log = append_log(log, f"Tab 3 (Semantic Kernel) horizon = {num_days} day(s)")
    log = append_log(log, f"Manual text truncated to {len(material_text)} characters")
    yield render_pipeline_html(SK_STEPS, active_index=0, done_upto=0), log, "", ""

    # ChatHistory holds the planner conversation (system + user + later the reply).
    chat_history = ChatHistory()
    chat_history.add_system_message(
        "You are ServiceIQ's preventive-maintenance planner. Ground every task in the manual excerpt."
    )
    chat_history.add_user_message(
        f"Manual excerpt:\n{material_text}\n\nPlan preventive maintenance for {num_days} day(s)."
    )
    log = append_log(log, "ChatHistory ready (system + user messages)")
    yield render_pipeline_html(SK_STEPS, active_index=1, done_upto=1), log, "", ""

    log = append_log(log, "Invoking semantic function MaintenancePlanner-generate_maintenance_schedule ...")
    yield render_pipeline_html(SK_STEPS, active_index=2, done_upto=2), log, "", ""
    plan = await sk_kernel.invoke(
        sk_schedule_fn,
        KernelArguments(material_text=material_text, num_days=str(num_days)),
    )
    plan_text = str(plan)
    chat_history.add_assistant_message(plan_text)   # keep the conversation traceable
    log = append_log(log, f"Plan returned ({len(plan_text)} chars); appended to ChatHistory")
    yield render_pipeline_html(SK_STEPS, active_index=3, done_upto=3), log, plan_text, ""

    # Native function — pure code, no LLM.
    work_orders = await sk_kernel.invoke(
        sk_kernel.plugins["MaintenancePlanner"]["mock_work_order_sync"],
        KernelArguments(plan_text=plan_text),
    )
    log = append_log(log, f"Native function created {str(work_orders).count('WO-')} mock work orders")
    yield render_pipeline_html(SK_STEPS, active_index=-1, done_upto=len(SK_STEPS)), log, plan_text, str(work_orders)

print("Tab 3 handler ready (async).")

---
## Task 4 — Haystack: grounded RAG pipeline

The most explicit framework: a `Pipeline` whose components are wired with `connect()`.

- **Indexing phase** (Build Index button): `DocumentSplitter(split_by="word",
  split_length=150, split_overlap=30)` → `OpenAIDocumentEmbedder` → `InMemoryDocumentStore`.
- **Query phase** (Ask button, run one component at a time so retrieval is visible):
  `OpenAITextEmbedder` → `InMemoryEmbeddingRetriever(top_k=3)` → `ChatPromptBuilder` →
  `OpenAIChatGenerator`.
- **Grounding prompt**: *answer only from the context, otherwise say you don't know.*
- Questions are **blocked** until at least one document is indexed.

In [ ]:
from haystack import Pipeline, Document
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.embedders import OpenAIDocumentEmbedder, OpenAITextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage
from haystack.document_stores.in_memory import InMemoryDocumentStore

hs_store = InMemoryDocumentStore()
hs_docs_indexed = False

hs_splitter = DocumentSplitter(split_by="word", split_length=150, split_overlap=30)
hs_doc_embedder = OpenAIDocumentEmbedder(model=OPENAI_EMBED_MODEL)
hs_text_embedder = OpenAITextEmbedder(model=OPENAI_EMBED_MODEL)
hs_retriever = InMemoryEmbeddingRetriever(document_store=hs_store, top_k=3)

HS_GROUNDED_TEMPLATE = [
    ChatMessage.from_user(
        "You are ServiceIQ, a field-service assistant for equipment documentation.\n"
        "Answer ONLY from the context below. If the context does not contain the answer, "
        "reply exactly: I don't know.\n\n"
        "Context:\n"
        "{% for doc in context %}\n"
        "{{ doc.content }}\n"
        "---\n"
        "{% endfor %}\n"
        "Technician question: {{ question }}"
    )
]
hs_prompt_builder = ChatPromptBuilder(
    template=HS_GROUNDED_TEMPLATE, required_variables=["context", "question"]
)
hs_generator = OpenAIChatGenerator(model=OPENAI_MODEL)

hs_pipeline = Pipeline()
hs_pipeline.add_component("text_embedder", hs_text_embedder)
hs_pipeline.add_component("retriever", hs_retriever)
hs_pipeline.add_component("prompt_builder", hs_prompt_builder)
hs_pipeline.add_component("generator", hs_generator)

hs_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
hs_pipeline.connect("retriever.documents", "prompt_builder.context")
hs_pipeline.connect("prompt_builder.prompt", "generator.messages")

try:
    hs_pipeline.draw(path="serviceiq_haystack_pipeline.png")
    print("Pipeline diagram saved to serviceiq_haystack_pipeline.png")
except Exception as e:
    print("Diagram skipped (graphviz not available):", e)
print(hs_pipeline)

In [ ]:
HS_INDEX_STEPS = ["Extract PDF", "Split (150 words, overlap 30)", "Embed chunks", "Write to store"]
HS_QUERY_STEPS = ["Embed question", "Retrieve top-3", "Build grounded prompt", "Generate answer"]

def hs_build_index(pdf_path=None):
    """Indexing phase: split -> embed -> write. Run once (or to refresh)."""
    global hs_docs_indexed
    path = pdf_path or MANUAL_PDF
    text = extract_pdf_text(path)
    docs = [Document(content=text)]
    split = hs_splitter.run(documents=docs)["documents"]
    embedded = hs_doc_embedder.run(documents=split)["documents"]
    hs_store.write_documents(embedded)
    hs_docs_indexed = True
    return (f"Indexed {len(embedded)} chunks from {path} "
            f"(split_length=150 words, split_overlap=30).")

def hs_ask(question, log):
    """Query phase, one component at a time, so retrieval is visible."""
    if not hs_docs_indexed:
        yield (render_pipeline_html(HS_QUERY_STEPS),
               append_log(log, "BLOCKED: no documents indexed yet — click Build Index first."),
               "", "")
        return
    if not question or not question.strip():
        yield (render_pipeline_html(HS_QUERY_STEPS),
               append_log(log, "No question entered."), "", "")
        return

    q = question.strip()
    log = append_log(log, f"Tab 4 (Haystack) received: {q!r}")
    yield render_pipeline_html(HS_QUERY_STEPS, active_index=0, done_upto=0), log, "", ""

    # 1) Embed the question
    q_embedding = hs_text_embedder.run(text=q)["embedding"]
    log = append_log(log, f"Question embedded ({len(q_embedding)} dims)")
    yield render_pipeline_html(HS_QUERY_STEPS, active_index=1, done_upto=1), log, "", ""

    # 2) Retrieve top-3 — show the passages BEFORE generating
    retrieved = hs_retriever.run(query_embedding=q_embedding, top_k=3)["documents"]
    passages = []
    for i, d in enumerate(retrieved, start=1):
        score = f"{d.score:.3f}" if getattr(d, "score", None) is not None else "n/a"
        passages.append(f"**Passage {i}** (score {score}):\n\n{d.content}")
    log = append_log(log, f"Retrieved {len(retrieved)} passages (top_k=3)")
    yield (render_pipeline_html(HS_QUERY_STEPS, active_index=2, done_upto=2), log,
           "### Retrieved passages\n\n" + "\n\n---\n\n".join(passages), "")

    # 3) Build the grounded prompt
    prompt_out = hs_prompt_builder.run(
        template=HS_GROUNDED_TEMPLATE, context=retrieved, question=q
    )
    log = append_log(log, f"Grounded prompt built ({sum(len(m.text) for m in prompt_out['prompt'])} chars)")
    yield render_pipeline_html(HS_QUERY_STEPS, active_index=3, done_upto=3), log, \
          "### Retrieved passages\n\n" + "\n\n---\n\n".join(passages), ""

    # 4) Generate — the grounding rule forces refusal when context is empty
    reply = hs_generator.run(messages=prompt_out["prompt"])["replies"][0]
    answer = reply.text if hasattr(reply, "text") else str(reply)
    log = append_log(log, "Answer generated under the grounding rule "
                          '(context-only, else "I don\'t know")')
    yield (render_pipeline_html(HS_QUERY_STEPS, active_index=-1, done_upto=len(HS_QUERY_STEPS)),
           log, "### Retrieved passages\n\n" + "\n\n---\n\n".join(passages), answer)

print("Tab 4 handlers ready. Run hs_build_index() or use the Build Index button.")

---
## Launch the ServiceIQ app (four tabs)

Each tab: inputs on the left, live execution panel + execution log on the right.
Test with the four sample queries — the last one is out of scope and must be refused in
Tabs 2 and 4:
1. "Dispenser shows error E-104 after a power cycle. What should I check first?"
2. "What is the recommended defrost interval for the display case?"
3. "List the lockout/tagout steps before replacing the pump diaphragm."
4. "What is the warranty period for the moon rover?"

In [ ]:
import gradio as gr

with gr.Blocks(title="ServiceIQ — RAG Assistant") as demo:
    gr.Markdown("# ServiceIQ — RAG Assistant for Field Technicians\n"
                "Four frameworks, one OpenAI backend. Answer only from the equipment manual.")

    # ---------------- Tab 1: LangChain ----------------
    with gr.Tab("1 · LangChain — Troubleshooting (LLM only)"):
        gr.Markdown("**No retrieval.** Shows what a bare LLM does — and why it can hallucinate.")
        with gr.Row():
            with gr.Column():
                lc_q = gr.Textbox(label="Technician question", lines=2,
                                  placeholder="Dispenser shows error E-104 after a power cycle...")
                lc_btn = gr.Button("Run chain", variant="primary")
            with gr.Column():
                lc_panel = gr.HTML(render_pipeline_html(LC_STEPS), label="Execution panel")
                lc_log = gr.Textbox(label="Execution log", lines=8, interactive=False)
        lc_out = gr.Textbox(label="Answer (numbered troubleshooting steps)", lines=12)
        lc_btn.click(run_langchain_tab, inputs=[lc_q, lc_panel, lc_log],
                     outputs=[lc_panel, lc_log, lc_out])

    # ---------------- Tab 2: LlamaIndex ----------------
    with gr.Tab("2 · LlamaIndex — Manual Q&A"):
        gr.Markdown("**Indexing phase** (Build Index, once) then **query phase** (Ask) with top-2 sources.")
        with gr.Row():
            with gr.Column():
                li_pdf = gr.File(label="Equipment manual (PDF)", file_types=[".pdf"])
                li_build_btn = gr.Button("Build Index", variant="secondary")
                li_status = gr.Textbox(label="Index status", lines=2, interactive=False)
                li_q = gr.Textbox(label="Technician question", lines=2)
                li_btn = gr.Button("Ask", variant="primary")
            with gr.Column():
                li_panel = gr.HTML(render_pipeline_html(LI_QUERY_STEPS), label="Execution panel")
                li_log = gr.Textbox(label="Execution log", lines=8, interactive=False)
        li_out = gr.Markdown(label="Answer + retrieved sources")
        li_build_btn.click(lambda f: build_li_index(getattr(f, "name", f) if f else None),
                           inputs=[li_pdf], outputs=[li_status])
        li_btn.click(ask_li, inputs=[li_q, li_log], outputs=[li_panel, li_log, li_out])

    # ---------------- Tab 3: Semantic Kernel ----------------
    with gr.Tab("3 · Semantic Kernel — Maintenance planner"):
        gr.Markdown("**Plugin** with a semantic function (plan) and a native function (mock work orders).")
        with gr.Row():
            with gr.Column():
                sk_days = gr.Slider(1, 7, value=3, step=1,
                                    label="Maintenance horizon (days)")
                sk_btn = gr.Button("Generate plan + work orders", variant="primary")
            with gr.Column():
                sk_panel = gr.HTML(render_pipeline_html(SK_STEPS), label="Execution panel")
                sk_log = gr.Textbox(label="Execution log", lines=8, interactive=False)
        sk_plan = gr.Textbox(label="Preventive-maintenance plan", lines=10)
        sk_orders = gr.Textbox(label="Mock work-order log (no real system)", lines=8)
        sk_btn.click(run_sk_tab, inputs=[sk_days, sk_panel, sk_log],
                     outputs=[sk_panel, sk_log, sk_plan, sk_orders])

    # ---------------- Tab 4: Haystack ----------------
    with gr.Tab("4 · Haystack — Grounded RAG"):
        gr.Markdown("**Explicit pipeline wired with connect().** Questions are blocked until "
                    "a document is indexed. Grounding rule: answer only from context, "
                    "otherwise say **I don't know**.")
        with gr.Row():
            with gr.Column():
                hs_pdf = gr.File(label="Equipment manual (PDF)", file_types=[".pdf"])
                hs_build_btn = gr.Button("Build Index", variant="secondary")
                hs_status = gr.Textbox(label="Index status", lines=2, interactive=False)
                hs_q = gr.Textbox(label="Technician question", lines=2)
                hs_btn = gr.Button("Ask (grounded)", variant="primary")
            with gr.Column():
                hs_panel = gr.HTML(render_pipeline_html(HS_QUERY_STEPS), label="Execution panel")
                hs_log = gr.Textbox(label="Execution log", lines=8, interactive=False)
        hs_passages = gr.Markdown(label="Retrieved passages")
        hs_out = gr.Textbox(label="Grounded answer", lines=6)
        hs_build_btn.click(lambda f: hs_build_index(getattr(f, "name", f) if f else None),
                           inputs=[hs_pdf], outputs=[hs_status])
        hs_btn.click(hs_ask, inputs=[hs_q, hs_log],
                     outputs=[hs_panel, hs_log, hs_passages, hs_out])

demo.launch()

---
## Grounding and testing checklist

Run each query in the relevant tab and record what happened:

| Sample query | Tab 1 (LangChain) | Tab 2 (LlamaIndex) | Tab 4 (Haystack) |
|---|---|---|---|
| E-104 after power cycle | Answers from pretraining — check whether the steps match Section 1.1 of the manual | Answers from retrieved Section 1.1 passages, sources shown | Answers from top-3 passages; passages shown before generation |
| Defrost interval for the display case | May guess; no guarantee it matches CR-220 specs | Grounded in Section 2.1 (every 6 hours, 20 min) | Grounded in Section 2.1 |
| Lockout/tagout before pump diaphragm | Generic LOTO steps, possibly incomplete | Grounded in Section 3.1 (7 ordered steps) | Grounded in Section 3.1 |
| "Warranty period for the moon rover?" | **Likely hallucinates** a warranty answer — no manual, no refusal rule | **Must refuse**: no moon rover content in the index | **Must refuse**: grounding prompt forces "I don't know" |

Also try the knobs: set `split_overlap=0` and/or `top_k=1` in Tab 4 and re-run the E-104
query — observe how answers degrade when chunk boundaries cut a procedure in half or
retrieval misses the passage containing it.

---
## Report — framework comparison (fill from your own runs)

| Criterion | LangChain | LlamaIndex | Semantic Kernel | Haystack |
|---|---|---|---|---|
| Lines of code for the core logic | *(fill: ~?)* | *(fill)* | *(fill)* | *(fill)* |
| Time to first working answer | *(fill)* | *(fill)* | *(fill)* | *(fill)* |
| Answered E-104 correctly? (yes/no) | *(fill — likely yes but ungrounded)* | *(fill)* | *(n/a — different task)* | *(fill)* |
| Refused the out-of-scope query? (yes/no) | *(fill — likely no)* | *(fill)* | *(n/a)* | *(fill)* |
| Could you see the retrieved context? | no (no retrieval) | yes (source_nodes) | no (text truncated, no retrieval) | yes (passages shown) |
| Ease of debugging (1–5) | *(fill)* | *(fill)* | *(fill)* | *(fill)* |
| One thing you liked | *(fill)* | *(fill)* | *(fill)* | *(fill)* |
| One limitation you hit | *(fill)* | *(fill)* | *(fill)* | *(fill)* |

### Reflection questions

**1. Why did Tab 1 answer the E-104 question even though it never saw the manual? Was the answer right?**
Tab 1's prompt gives the model a field-service role, and E-104-style flow-meter faults are
generic enough that the model produces confident, plausible steps from pretraining
knowledge. It never saw the manual, so it could not know the DX-500-specific sequence
(voltage check → J4 connector → 60 s/90 s controlled power cycle → calibration routine
with a 20-litre canister → pulse sensor board DX-PM-114). Whether it is "right" is
unverifiable from the app itself — that is the point: an ungrounded answer *looks*
authoritative but carries no traceability, which is unacceptable for safety procedures.

**2. What changed when you set split_overlap to 0 or top_k to 1?**
With `split_overlap=0`, a procedure that straddles a 150-word boundary (e.g. the LOTO
steps) gets cut in half: no chunk contains both halves, so retrieval returns a fragment
and the answer drops steps. With `top_k=1`, only one passage reaches the prompt — if the
best-scoring chunk holds only part of the answer, the model cannot recover the rest and
may refuse or under-answer. Overlap and top-k are insurance against boundary and
ranking errors; the cost is more tokens and some duplicated context.

**3. Which framework would you choose for ServiceIQ in production, and why?**
**Haystack** for the core answering pipeline: every stage (splitter, embedders, retriever,
prompt builder, generator) is an explicit, swappable component wired with `connect()`,
retrieval is inspectable before generation, and the grounding/refusal behavior is
prompt-controlled and testable — exactly what a safety-critical industrial assistant
needs for auditing and regression testing. I would keep **Semantic Kernel** alongside it
for the business workflow (native + semantic functions driving work-order systems), and
use **LlamaIndex**'s indexing abstractions for fast prototyping on new manuals. LangChain
remains useful for agents and query structuring, but for grounded answers its lower
transparency is a liability here.